# Listing Weeks 4-5

In [3]:
import pandas as pd

listing = pd.read_csv("../../IDX_Data/listing_with_rates.csv")
print("Initial shape:", listing.shape)
listing.head()

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_54517/918148656.py:3: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listing = pd.read_csv("../../IDX_Data/listing_with_rates.csv")


Initial shape: (540624, 86)


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,year_month,rate_30yr_fixed
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,NaN,NaN,90067,NaN,2105.00,177861.0,NaN,2220 Avenue Of The Stars 2704,2024-01,6.6425
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,3.0,Capistrano Unified,92677,NaN,254.00,5300.0,NaN,16 Palisades,2024-01,6.6425
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road,2024-01,6.6425
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,4.0,Walnut Valley Unified,91765,NaN,295.95,58232.0,NaN,2250 Indian Creek Road,2024-01,6.6425
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,2.0,Newport Mesa Unified,92662,NaN,0.00,2250.0,NaN,317 E. Bayfront,2024-01,6.6425


In [4]:
date_cols = [
    'ListingContractDate',
    'PurchaseContractDate',
    'CloseDate',
    'ContractStatusChangeDate'
]

for col in date_cols:
    if col in listing.columns:
        listing[col] = pd.to_datetime(listing[col], errors='coerce')

listing[date_cols].head()

,ListingContractDate,PurchaseContractDate,CloseDate,ContractStatusChangeDate
0,2024-01-01,2024-05-07,NaT,2024-05-07
1,2024-01-24,NaT,NaT,2024-01-24
2,2024-01-12,NaT,NaT,2024-01-12
3,2024-01-20,NaT,NaT,2024-01-20
4,2024-01-12,NaT,NaT,2024-01-12


In [5]:
missing_pct = listing.isnull().mean() * 100
high_missing_cols = missing_pct[missing_pct > 90].index.tolist()

print("Columns >90% missing:")
print(high_missing_cols)

Columns >90% missing:
['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'BuilderName', 'TaxYear', 'BuildingAreaTotal', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']


In [6]:
before_cols = listing.shape[1]
listing = listing.drop(columns=high_missing_cols)
after_cols = listing.shape[1]

print("Columns before:", before_cols)
print("Columns after:", after_cols)

Columns before: 86
Columns after: 73


In [7]:
numeric_cols = [
    'ListPrice',
    'LivingArea',
    'DaysOnMarket',
    'BedroomsTotal',
    'BathroomsTotalInteger'
]

for col in numeric_cols:
    if col in listing.columns:
        listing[col] = pd.to_numeric(listing[col], errors='coerce')

listing[numeric_cols].dtypes

ListPrice                float64
LivingArea               float64
DaysOnMarket               int64
BedroomsTotal            float64
BathroomsTotalInteger    float64
dtype: object

In [8]:
before_rows = listing.shape[0]
listing = listing[
    (listing['ListPrice'] > 0) &
    (listing['LivingArea'] > 0) &
    (listing['DaysOnMarket'] >= 0) &
    (listing['BedroomsTotal'] >= 0) &
    (listing['BathroomsTotalInteger'] >= 0)
]
after_rows = listing.shape[0]

print("Rows before:", before_rows)
print("Rows after:", after_rows)

Rows before: 540624
Rows after: 539597


In [9]:
listing['listing_after_close_flag'] = (
    listing['ListingContractDate'] > listing['CloseDate']
)

listing['purchase_after_close_flag'] = (
    listing['PurchaseContractDate'] > listing['CloseDate']
)

listing['negative_timeline_flag'] = (
    (listing['ListingContractDate'] > listing['PurchaseContractDate']) |
    (listing['PurchaseContractDate'] > listing['CloseDate'])
)

In [10]:
print("Listing after close:", listing['listing_after_close_flag'].sum())
print("Purchase after close:", listing['purchase_after_close_flag'].sum())
print("Negative timeline:", listing['negative_timeline_flag'].sum())

Listing after close: 71
Purchase after close: 263
Negative timeline: 529


In [11]:
missing_coords = listing['Latitude'].isnull() | listing['Longitude'].isnull()
zero_coords = (listing['Latitude'] == 0) | (listing['Longitude'] == 0)
positive_longitude = listing['Longitude'] > 0

print("Missing coords:", missing_coords.sum())
print("Zero coords:", zero_coords.sum())
print("Positive longitude:", positive_longitude.sum())

Missing coords: 80133
Zero coords: 60
Positive longitude: 74


In [12]:
listing.to_csv("listing_cleaned_week4_5.csv", index=False)
print("Cleaned LISTING dataset saved.")

Cleaned LISTING dataset saved.


### Data Cleaning Summary

- Converted date fields to datetime format
- Removed columns with more than 90% missing values
- Ensured numeric fields were properly typed
- Removed invalid numeric values (e.g., zero or negative values)
- Created date consistency flags
- Performed geographic data quality checks

### Validation Summary

- Row counts before and after filtering were tracked
- Date inconsistencies were identified using flag variables
- Geographic anomalies were flagged (missing or invalid coordinates)
- Final dataset is cleaned and ready for analysis